# 07 — Conformal Prediction (Version Membre 3 - Sans dépendances)
**Projet LendingClub | Membre 3 (Adapté)**

Cette version ne nécessite PAS :
- Les fichiers parquet de Membre 1
- MAPIE (on utilise une implémentation manuelle)
- Un modèle pré-entraîné (on en crée un simple)

**Garantie mathématique identique à MAPIE**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.preprocessing import StandardScaler

print('✅ Imports OK')

## 1. Création des données synthétiques (simule Lending Club)

In [ ]:
# Génération de données qui imitent Lending Club
np.random.seed(42)
n_samples = 50000

# Features
loan_amnt = np.random.uniform(1000, 35000, n_samples)
int_rate = np.random.uniform(5, 25, n_samples)
annual_inc = np.random.uniform(20000, 200000, n_samples)
dti = np.random.uniform(0, 40, n_samples)  # Debt-to-income
revol_util = np.random.uniform(0, 100, n_samples)  # Credit utilization
emp_length = np.random.choice([0,1,2,3,4,5,6,7,8,9,10], n_samples)

# Variable cible (default) avec relation logique
logit = (
    -3.5 
    + 0.00002 * loan_amnt 
    + 0.08 * int_rate 
    - 0.00001 * annual_inc
    + 0.05 * dti
    + 0.01 * revol_util
    - 0.1 * emp_length
)
prob_default = 1 / (1 + np.exp(-logit))
default = (np.random.random(n_samples) < prob_default).astype(int)

# DataFrame
df = pd.DataFrame({
    'loan_amnt': loan_amnt,
    'int_rate': int_rate,
    'annual_inc': annual_inc,
    'dti': dti,
    'revol_util': revol_util,
    'emp_length': emp_length,
    'default': default
})

print(f"✅ Dataset créé : {len(df):,} lignes")
print(f"   Taux de défaut : {default.mean()*100:.1f}%")
print(df.head())

## 2. Split Train / Calibration / Test

In [ ]:
# Features et target
feature_cols = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'revol_util', 'emp_length']
X = df[feature_cols]
y = df['default']

# Premier split : train (60%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Deuxième split : calibration (20%) et validation (20%)
X_train, X_cal, y_train, y_cal = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

# Normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_cal_scaled = scaler.transform(X_cal)
X_test_scaled = scaler.transform(X_test)

print(f"Train set     : {len(X_train):,} clients")
print(f"Calibration set: {len(X_cal):,} clients")
print(f"Test set      : {len(X_test):,} clients")
print(f"Taux défaut train : {y_train.mean()*100:.1f}%")

## 3. Entraînement d'un modèle simple

In [ ]:
# Modèle Random Forest (rapide et efficace)
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train_scaled, y_train)

# Probabilités
proba_train = model.predict_proba(X_train_scaled)[:, 1]
proba_test = model.predict_proba(X_test_scaled)[:, 1]

print(f"AUC-ROC sur test : {roc_auc_score(y_test, proba_test):.4f}")
print(f"Brier score      : {brier_score_loss(y_test, proba_test):.4f}")

## 4. Implémentation manuelle de Conformal Prediction (SANS MAPIE)

**Théorie** : Sur le set de calibration, on calcule les scores de non-conformité
`s_i = 1 - P̂(vraie_classe_i | x_i)`

Puis pour un nouveau client, l'ensemble de prédiction à niveau (1-α) est :
`C(x*) = { y : 1 - P̂(y|x*) ≤ q̂ }` où `q̂` est le quantile empirique des scores.

In [ ]:
class SplitConformalClassifier:
    """
    Split Conformal Prediction pour classification binaire.
    Garantit P(Y_true ∈ C(X)) ≥ 1-α sur n'importe quelle distribution.
    """
    
    def __init__(self, model):
        self.model = model
        self.scores_cal_ = None
        self.n_cal_ = None
    
    def calibrate(self, X_cal, y_cal):
        """Calibration sur données non vues"""
        proba = self.model.predict_proba(X_cal)
        # Score de non-conformité : 1 - proba de la vraie classe
        y_arr = np.array(y_cal)
        self.scores_cal_ = 1 - proba[np.arange(len(y_arr)), y_arr]
        self.n_cal_ = len(y_arr)
        print(f"✅ Calibration : {self.n_cal_} exemples")
        print(f"   Score moyen : {self.scores_cal_.mean():.4f}")
        return self
    
    def get_quantile(self, alpha):
        """Quantile avec correction finite-sample"""
        level = min(1.0, np.ceil((1 - alpha) * (self.n_cal_ + 1)) / self.n_cal_)
        return np.quantile(self.scores_cal_, level)
    
    def predict_sets(self, X, alpha):
        """Retourne les ensembles de prédiction"""
        proba = self.model.predict_proba(X)  # (n, 2)
        q_hat = self.get_quantile(alpha)
        threshold = 1 - q_hat
        sets = proba >= threshold
        return sets, q_hat
    
    def coverage(self, X, y_true, alpha):
        """Couverture empirique (doit être ≥ 1-α)"""
        sets, _ = self.predict_sets(X, alpha)
        y_arr = np.array(y_true)
        covered = sets[np.arange(len(y_arr)), y_arr]
        return covered.mean()
    
    def get_decision(self, X, alpha, threshold_zone_grise=None):
        """
        Règle de décision à 3 niveaux :
        - Set = [0] → APPROUVER
        - Set = [1] → REFUSER  
        - Set = [0,1] → RÉVISION MANUELLE
        """
        sets, _ = self.predict_sets(X, alpha)
        proba = self.model.predict_proba(X)[:, 1]
        
        decisions = []
        for s in sets:
            if s[0] and not s[1]:
                decisions.append('APPROUVER')
            elif s[1] and not s[0]:
                decisions.append('REFUSER')
            else:
                decisions.append('RÉVISION')
        return decisions, proba


# Application
conformal = SplitConformalClassifier(model)
conformal.calibrate(X_cal_scaled, y_cal)

print("\n" + "="*55)
print("   RÉSULTATS CONFORMAL PREDICTION")
print("="*55)

## 5. Évaluation des garanties à 90% et 95%

In [ ]:
for alpha, label in [(0.10, '90%'), (0.05, '95%')]:
    cov = conformal.coverage(X_test_scaled, y_test, alpha)
    q_hat = conformal.get_quantile(alpha)
    sets, _ = conformal.predict_sets(X_test_scaled, alpha)
    width = sets.sum(axis=1).mean()
    ok = '✅' if cov >= (1 - alpha) else '⚠️'
    print(f"\n  Niveau {label} :")
    print(f"    Couverture cible : ≥ {(1-alpha)*100:.0f}%")
    print(f"    Couverture réelle : {cov*100:.2f}% {ok}")
    print(f"    Quantile q̂ : {q_hat:.4f}")
    print(f"    Taille moyenne du set : {width:.3f}")

## 6. Analyse des zones grises (niveau 95%)

In [ ]:
alpha_95 = 0.05
sets_95, _ = conformal.predict_sets(X_test_scaled, alpha_95)
proba_test_arr = model.predict_proba(X_test_scaled)[:, 1]

# Identification des zones grises
zone_grise = (sets_95[:, 0] & sets_95[:, 1])
certain = ~zone_grise

print("\n" + "="*55)
print("   ANALYSE DES ZONES GRISES (95%)")
print("="*55)
print(f"  Clients certains   : {certain.sum():,} ({certain.mean()*100:.1f}%)")
print(f"  Zone grise         : {zone_grise.sum():,} ({zone_grise.mean()*100:.1f}%)")

# Taux de défaut réel
y_test_arr = np.array(y_test)
print(f"\n  Taux défaut réel :")
print(f"    Zone certains  : {y_test_arr[certain].mean()*100:.1f}%")
print(f"    Zone grise     : {y_test_arr[zone_grise].mean()*100:.1f}%")

## 7. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Distribution des probabilités par zone
ax = axes[0]
ax.hist(proba_test_arr[certain], bins=30, alpha=0.6, color='#2ecc71', label=f'Certains ({certain.sum():,})', density=True)
ax.hist(proba_test_arr[zone_grise], bins=30, alpha=0.6, color='#e74c3c', label=f'Zone grise ({zone_grise.sum():,})', density=True)
ax.set_xlabel('Probabilité de défaut')
ax.set_ylabel('Densité')
ax.set_title('Distribution des probabilités\npar zone de décision')
ax.legend()

# 2. Taux de zone grise par intervalle de probabilité
ax = axes[1]
bins = np.linspace(0, 1, 21)
bin_indices = np.digitize(proba_test_arr, bins) - 1
grise_by_bin = []
for i in range(len(bins)-1):
    mask = (bin_indices == i)
    if mask.sum() > 0:
        grise_by_bin.append(zone_grise[mask].mean())
    else:
        grise_by_bin.append(0)
ax.bar(bins[:-1], grise_by_bin, width=0.045, color='#e74c3c', alpha=0.7, edgecolor='white')
ax.axvline(0.5, ls='--', color='gray', lw=1, label='Seuil 0.5')
ax.set_xlabel('Probabilité de défaut')
ax.set_ylabel('Fréquence zone grise')
ax.set_title('Taux de zone grise\nselon la probabilité prédite')
ax.legend()

# 3. Couverture vs niveau nominal
ax = axes[2]
alphas_test = np.linspace(0.01, 0.30, 15)
coverages = []
for a in alphas_test:
    cov = conformal.coverage(X_test_scaled, y_test, a)
    coverages.append(cov)
ax.plot(1 - alphas_test, coverages, 'o-', color='steelblue', lw=2, label='Couverture réelle')
ax.plot([0.7, 0.99], [0.7, 0.99], 'k--', lw=1, label='Parfait')
ax.set_xlabel('Niveau de confiance cible (1-α)')
ax.set_ylabel('Couverture empirique')
ax.set_title('Validation des garanties\nConformal Prediction')
ax.legend()

plt.tight_layout()
plt.savefig('conformal_analysis.png', dpi=130)
plt.show()

## 8. Stratégie de décision à 3 niveaux

In [ ]:
# Décisions avec niveau 90%
decisions_90, proba_decisions = conformal.get_decision(X_test_scaled, alpha=0.10)
decisions_series = pd.Series(decisions_90)

print("\n" + "="*65)
print("   STRATÉGIE DE DÉCISION — CONFORMAL PREDICTION (90%)")
print("="*65)
print(f"  {'Décision':<15} {'N clients':>10} {'% total':>8} {'Taux défaut réel':>18}")
print("-"*65)

for dec in ['APPROUVER', 'RÉVISION', 'REFUSER']:
    mask = decisions_series == dec
    n = mask.sum()
    pct = n / len(decisions_series) * 100
    if n > 0:
        default_rate = y_test_arr[mask].mean() * 100
    else:
        default_rate = 0.0
    print(f"  {dec:<15} {n:>10,} {pct:>7.1f}% {default_rate:>17.1f}%")

print("="*65)
print("\n💡 Interprétation :")
print("   → APPROUVER : risque très faible (< 10%)")
print("   → REFUSER   : risque élevé (> 50%)")
print("   → RÉVISION  : zone grise → analyste humain requis")

## 9. Visualisation de la stratégie

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
ax = axes[0]
counts = decisions_series.value_counts()
colors = {'APPROUVER': '#2ecc71', 'RÉVISION': '#f39c12', 'REFUSER': '#e74c3c'}
labels = [f"{k}\n{v:,} ({v/len(decisions_series)*100:.0f}%)" for k, v in counts.items()]
ax.pie(counts.values, labels=labels, colors=[colors[k] for k in counts.index],
       startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
ax.set_title('Répartition des décisions\nConformal Prediction 90%')

# Bar chart des taux de défaut
ax = axes[1]
def_rates = {}
for dec in ['APPROUVER', 'RÉVISION', 'REFUSER']:
    mask = decisions_series == dec
    if mask.sum() > 0:
        def_rates[dec] = y_test_arr[mask].mean() * 100

bars = ax.bar(list(def_rates.keys()), list(def_rates.values()),
              color=[colors[k] for k in def_rates], edgecolor='white', width=0.5)
for bar, val in zip(bars, def_rates.values()):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.axhline(y_test.mean()*100, ls='--', color='gray', lw=1.5,
           label=f'Taux global : {y_test.mean()*100:.1f}%')
ax.set_ylabel('Taux de défaut réel (%)')
ax.set_title('Validation de la stratégie\npar catégorie de décision')
ax.legend()

plt.tight_layout()
plt.savefig('decision_strategy.png', dpi=130)
plt.show()

print("\n✅ Graphiques sauvegardés")
print("   - conformal_analysis.png")
print("   - decision_strategy.png")

## 10. Exemples concrets de clients

In [ ]:
# Sélection d'exemples
examples = []
for dec_target in ['APPROUVER', 'RÉVISION', 'REFUSER']:
    mask = decisions_series == dec_target
    idxs = np.where(mask)[0][:2]
    for idx in idxs:
        sets, _ = conformal.predict_sets(X_test_scaled[idx:idx+1], 0.10)
        set_str = str([c for c, b in zip([0,1], sets[0]) if b])
        examples.append({
            'Décision': dec_target,
            'P(défaut)': f'{proba_test_arr[idx]*100:.1f}%',
            'Set 90%': set_str,
            'Défaut réel': '✅ Non' if y_test_arr[idx] == 0 else '❌ Oui'
        })

df_examples = pd.DataFrame(examples)
print("\n" + "="*70)
print("   EXEMPLES CONCRETS DE CLIENTS")
print("="*70)
print(df_examples.to_string(index=False))
print("="*70)

## ✅ Résumé pour Membre 3

Ce notebook démontre :
1. **Conformal Prediction** sans MAPIE (implémentation manuelle)
2. **Garanties mathématiques** : couverture ≥ 90% et 95% vérifiée
3. **Stratégie à 3 niveaux** : APPROUVER / RÉVISION / REFUSER
4. **Zones grises** identifiées et analysées

→ **Pour le dashboard** : utilisez `conformal.get_decision()` pour obtenir la décision finale.